# Zomato Bangalore Restaurants - Data Cleaning

## 1. Import Libraries

In [8]:
import numpy as np
import pandas as pd

## 2. Load Dataset

In [9]:
df = pd.read_csv('data/zomato.csv')
df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


## 3. Data Inspection

In [10]:
df.shape

(51717, 17)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   url                          51717 non-null  str  
 1   address                      51717 non-null  str  
 2   name                         51717 non-null  str  
 3   online_order                 51717 non-null  str  
 4   book_table                   51717 non-null  str  
 5   rate                         43942 non-null  str  
 6   votes                        51717 non-null  int64
 7   phone                        50509 non-null  str  
 8   location                     51696 non-null  str  
 9   rest_type                    51490 non-null  str  
 10  dish_liked                   23639 non-null  str  
 11  cuisines                     51672 non-null  str  
 12  approx_cost(for two people)  51371 non-null  str  
 13  reviews_list                 51717 non-null  str  
 14  m

In [12]:
df.describe()

,votes
count,51717.000000
mean,283.697527
std,803.838853
min,0.000000
25%,7.000000
50%,41.000000
75%,198.000000
max,16832.000000


## 4. Check Missing Values

In [13]:
missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percentage.round(2)
})

missing_summary.sort_values(by='Missing Count', ascending=False)

,Missing Count,Percentage (%)
dish_liked,28078,54.29
rate,7775,15.03
phone,1208,2.34
approx_cost(for two people),346,0.67
rest_type,227,0.44
cuisines,45,0.09
location,21,0.04
url,0,0.00
address,0,0.00
votes,0,0.00


## 5. Text Normalization and Duplicate Handling

In [14]:
zomato = df.copy()

text_cols = ['name', 'address', 'location', 'rest_type', 'cuisines', 'listed_in(type)', 'listed_in(city)']
for col in text_cols:
    zomato[col] = zomato[col].str.strip()

In [15]:
non_url_cols = [col for col in zomato.columns if col != 'url']

print("Exact duplicate rows (all columns):", zomato.duplicated().sum())
print("Duplicate rows excluding URL before removal:", zomato.duplicated(subset=non_url_cols).sum())
print("Rows with matching name and address:", zomato.duplicated(subset=['name', 'address']).sum())

Exact duplicate rows (all columns): 0
Duplicate rows excluding URL before removal: 40
Rows with matching name and address: 39218


Restaurants frequently appear under multiple delivery zones (listed_in(city)) and service types (listed_in(type)). These represent distinct listings for different services rather than duplicate errors, so we do not delete rows simply because restaurant names or locations match. Only confirmed duplicates across all features (excluding URL) are removed.

In [16]:
zomato = zomato.drop_duplicates(subset=non_url_cols).reset_index(drop=True)

print("Duplicate rows excluding URL after removal:", zomato.duplicated(subset=non_url_cols).sum())
print("Cleaned dataset shape:", zomato.shape)

Duplicate rows excluding URL after removal: 0
Cleaned dataset shape: (51677, 17)


## 6. Numeric Feature Cleaning

In [17]:
zomato['rate_original'] = zomato['rate']

clean_rate = (
    zomato['rate']
    .astype(str)
    .str.replace('/5', '', regex=False)
    .str.strip()
)
clean_rate = clean_rate.replace(['NEW', '-', 'nan'], np.nan)
zomato['rate'] = pd.to_numeric(clean_rate, errors='coerce')

zomato['rate'].describe()

count    41640.000000
mean         3.700454
std          0.440554
min          1.800000
25%          3.400000
50%          3.700000
75%          4.000000
max          4.900000
Name: rate, dtype: float64

In [18]:
cost_col = 'approx_cost(for two people)'
clean_cost = (
    zomato[cost_col]
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
clean_cost = clean_cost.replace(['nan', 'None'], np.nan)
zomato[cost_col] = pd.to_numeric(clean_cost, errors='coerce')

zomato[cost_col].describe()

count    51332.000000
mean       555.441537
std        438.881726
min         40.000000
25%        300.000000
50%        400.000000
75%        650.000000
max       6000.000000
Name: approx_cost(for two people), dtype: float64

## 7. Review and Validate Cleaning Results

In [19]:
print("Rate data type:", zomato['rate'].dtype)
print("Rate missing values:", zomato['rate'].isnull().sum())
print("Rate range:", zomato['rate'].min(), "to", zomato['rate'].max())

print("\nCost data type:", zomato[cost_col].dtype)
print("Cost missing values:", zomato[cost_col].isnull().sum())
print("Cost range:", zomato[cost_col].min(), "to", zomato[cost_col].max())

invalid_ratings = zomato[(zomato['rate'] < 1.0) | (zomato['rate'] > 5.0)]
invalid_costs = zomato[zomato[cost_col] <= 0]
invalid_votes = zomato[zomato['votes'] < 0]

print("\nInvalid ratings count (<1 or >5):", len(invalid_ratings))
print("Invalid costs count (<=0):", len(invalid_costs))
print("Invalid votes count (<0):", len(invalid_votes))

Rate data type: float64
Rate missing values: 10037
Rate range: 1.8 to 4.9

Cost data type: float64
Cost missing values: 345
Cost range: 40.0 to 6000.0

Invalid ratings count (<1 or >5): 0
Invalid costs count (<=0): 0
Invalid votes count (<0): 0


## 8. Before and After Validation

In [20]:
before_after = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Exact Duplicate Rows',
        'Confirmed Duplicates (excl. URL)',
        'Rate Column Dtype',
        'Rate Missing / Non-Numeric',
        'Cost Column Dtype',
        'Cost Missing Count',
        'Invalid Ratings (<1 or >5)',
        'Invalid Costs (<=0)'
    ],
    'Before Cleaning (df)': [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df.duplicated(subset=non_url_cols).sum(),
        str(df['rate'].dtype),
        df['rate'].isnull().sum() + (df['rate'] == 'NEW').sum() + (df['rate'] == '-').sum(),
        str(df[cost_col].dtype),
        df[cost_col].isnull().sum(),
        'N/A (text format)',
        'N/A (text format)'
    ],
    'After Cleaning (zomato)': [
        len(zomato),
        zomato.shape[1],
        zomato.duplicated().sum(),
        zomato.duplicated(subset=non_url_cols).sum(),
        str(zomato['rate'].dtype),
        zomato['rate'].isnull().sum(),
        str(zomato[cost_col].dtype),
        zomato[cost_col].isnull().sum(),
        len(invalid_ratings),
        len(invalid_costs)
    ]
})

before_after

,Metric,Before Cleaning (df),After Cleaning (zomato)
0,Total Rows,51717,51677
1,Total Columns,17,18
2,Exact Duplicate Rows,0,0
3,Confirmed Duplicates (excl. URL),40,0
4,Rate Column Dtype,str,float64
5,Rate Missing / Non-Numeric,10052,10037
6,Cost Column Dtype,str,float64
7,Cost Missing Count,346,345
8,Invalid Ratings (<1 or >5),N/A (text format),0
9,Invalid Costs (<=0),N/A (text format),0


In [21]:
missing_comparison = pd.DataFrame({
    'Before Count': df.isnull().sum(),
    'Before (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'After Count': zomato[df.columns].isnull().sum(),
    'After (%)': (zomato[df.columns].isnull().sum() / len(zomato) * 100).round(2)
})

missing_comparison[missing_comparison['Before Count'] > 0]

,Before Count,Before (%),After Count,After (%)
rate,7775,15.03,10037,19.42
phone,1208,2.34,1207,2.34
location,21,0.04,21,0.04
rest_type,227,0.44,227,0.44
dish_liked,28078,54.29,28049,54.28
cuisines,45,0.09,45,0.09
approx_cost(for two people),346,0.67,345,0.67


# 9. Preprocessing and Feature Engineering

## 9.1 Inspecting the cleaned data

In [34]:
cleaned_zomato = zomato.copy()

print("Dataset shape:", cleaned_zomato.shape)

Dataset shape: (51677, 18)


In [38]:
cleaned_zomato.info()

<class 'pandas.DataFrame'>
RangeIndex: 51677 entries, 0 to 51676
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   url                          51677 non-null  str    
 1   address                      51677 non-null  str    
 2   name                         51677 non-null  str    
 3   online_order                 51677 non-null  str    
 4   book_table                   51677 non-null  str    
 5   rate                         41640 non-null  float64
 6   votes                        51677 non-null  int64  
 7   phone                        50470 non-null  str    
 8   location                     51656 non-null  str    
 9   rest_type                    51450 non-null  str    
 10  dish_liked                   23628 non-null  str    
 11  cuisines                     51632 non-null  str    
 12  approx_cost(for two people)  51332 non-null  float64
 13  reviews_list               

In [36]:
print("\nMissing values:")
missing = pd.DataFrame({
    "Missing Count": cleaned_zomato.isnull().sum(),
    "Missing Percentage": (
        cleaned_zomato.isnull().mean() * 100
    ).round(2)
})
print(missing[missing["Missing Count"] > 0]
      .sort_values("Missing Percentage", ascending=False))


Missing values:
                             Missing Count  Missing Percentage
dish_liked                           28049               54.28
rate                                 10037               19.42
rate_original                         7767               15.03
phone                                 1207                2.34
approx_cost(for two people)            345                0.67
rest_type                              227                0.44
cuisines                                45                0.09
location                                21                0.04


In [37]:
print("\nNumber of unique values:")
print(cleaned_zomato.nunique().sort_values())


Number of unique values:
online_order                       2
book_table                         2
listed_in(type)                    7
listed_in(city)                   30
rate                              31
rate_original                     64
approx_cost(for two people)       70
location                          93
rest_type                         93
votes                           2328
cuisines                        2723
dish_liked                      5271
name                            8792
menu_item                       9098
address                        11495
phone                          14926
reviews_list                   22513
url                            51677
dtype: int64


## 9.2 Handling Missing Values

In [48]:
print("\nMissing values:")
missing = pd.DataFrame({
    "Missing Count": cleaned_zomato.isnull().sum(),
    "Missing Percentage": (
        cleaned_zomato.isnull().mean() * 100
    ).round(2)
})
print(missing[missing["Missing Count"] > 0]
      .sort_values("Missing Percentage", ascending=False))


Missing values:
                             Missing Count  Missing Percentage
dish_liked                           28049               54.28
rate                                 10037               19.42
rate_original                         7767               15.03
phone                                 1207                2.34
approx_cost(for two people)            345                0.67


In [ ]:
categorical_missing_cols = [
    "rest_type",
    "cuisines",
    "location"
]

for col in categorical_missing_cols:
    if col in cleaned_zomato.columns:
        cleaned_zomato[col] = cleaned_zomato[col].fillna("Unknown")

print(cleaned_zomato[categorical_missing_cols].isnull().sum())

rest_type    0
cuisines     0
location     0
dtype: int64


In [49]:
cleaned_zomato["dish_liked"] = (
    cleaned_zomato["dish_liked"].fillna("Unknown")
)

- Missing values in the rest_type, cuisines, location, and dish_liked columns were replaced with "Unknown".

## 9.3 Feature Engineering

### Feature 1: Cuisine Count
- This feature counts the number of cuisines a restaurant has, which may later be used to explain the differences in ratings between restaurants

In [64]:
cleaned_zomato["cuisine_count"] = (
    cleaned_zomato["cuisines"]
    .apply(lambda x: 0 if pd.isna(x) or x == "Unknown"
           else len(x.split(",")))
)

### Feature 2: Restaurant Type Count
- This feature counts the number of restaurant types listed for a restaurant.
- This will help us see whether a restaurant belongs to one type or multiple types, which may help the model identify differences between restaurants.

In [65]:
cleaned_zomato["rest_type_count"] = (
    cleaned_zomato["rest_type"]
    .apply(lambda x: 0 if pd.isna(x) or x == "Unknown"
           else len(x.split(",")))
)

### Feature 3: Cost Per Person
- This feature calculates the approximate cost per person for each restaurant, which will allow us to investigate whether dining cost is related to ratings.

In [66]:
cost_col = "approx_cost(for two people)"

cleaned_zomato["cost_per_person"] = (
    pd.to_numeric(cleaned_zomato[cost_col], errors="coerce") / 2
)

In [70]:

display(
    cleaned_zomato[
        [
            "name",
            "cuisines",
            "cuisine_count",
            "rest_type",
            "rest_type_count",
            cost_col,
            "cost_per_person"
        ]
    ].sample(10, random_state=42)
)


,name,cuisines,cuisine_count,rest_type,rest_type_count,approx_cost(for two people),cost_per_person
884,#refuel,"Cafe, Beverages",2,Cafe,1,400.0,200.0
33264,Cool Bites,"Fast Food, Juices",2,Quick Bites,1,200.0,100.0
6420,Khally Wally,"Biryani, Kebab",2,Quick Bites,1,250.0,125.0
10798,Chinese Corner,Chinese,1,Quick Bites,1,300.0,150.0
33631,Roundup Cafe,"Cafe, Continental, Beverages",3,Cafe,1,800.0,400.0
21095,Parivar,"North Indian, Chinese",2,Casual Dining,1,500.0,250.0
20870,Ibaco,Desserts,1,Dessert Parlor,1,300.0,150.0
8925,On The Nose,"Cafe, Italian, Mexican, Chinese",4,Cafe,1,600.0,300.0
28501,Punjabi Nawabi,North Indian,1,Casual Dining,1,500.0,250.0
24759,Indie Fusion Multi Cuisine Restaurant,"Fast Food, BBQ, Rolls, Arabian, North Indian",5,Quick Bites,1,400.0,200.0


The engineered features were inspected by comparing the original restaurant information with the newly created features. This helps verify that the cuisine count, restaurant type count, and estimated cost per person are calculated correctly and that the new features represent the original data as intended.